In [ ]:
pip install pandas

In [ ]:
### Upload CSV File ###
import pandas as pd
df = pd.read_csv('creditcard.csv')

In [ ]:
print(df['Class'].unique()) # Check the Unique class labels, if Nan value, Fill NaN values in 'Class' column with 0 (assuming 0 is the non-fraud label)

In [ ]:
X = df.drop('Class', axis=1)
y = df['Class']

In [ ]:
# Since dataset used has high imbalance between fraud and non-fraud labels, resampling it for comparison
from imblearn.combine import SMOTEENN
from imblearn.under_sampling import EditedNearestNeighbours
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(y_train.value_counts())
# Apply SMOTE + ENN
#smote = SMOTE(k_neighbors=1, random_state=42) if not enough minor samples
smote_enn = SMOTEENN( random_state=42)
X_res, y_res = smote_enn.fit_resample(X_train, y_train)

print(f"Resampled class distribution: {y_res.value_counts()}")


In [ ]:
# Applying Random Forest Classifier model 
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# Initialize and train Random Forest Classifier
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_res, y_res)

# Predict on the test set
y_pred = rf_model.predict(X_test)

# Classification report
print("Random Forest Classification Report:")
print(classification_report(y_test, y_pred))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)

# Plot confusion matrix using seaborn heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=['Non-Fraud', 'Fraud'], yticklabels=['Non-Fraud', 'Fraud'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Random Forest Confusion Matrix')
plt.show()


In [ ]:
# for checking valuable fetaures that can be used only
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier


# Train the Random Forest model
rf = RandomForestClassifier(
    n_estimators=200,
    class_weight='balanced',
    min_samples_leaf=10,
    random_state=42
)
rf.fit(X_res, y_res)

# Get feature importances
importances = rf.feature_importances_
feature_names = X_res.columns

# Create a DataFrame for visualization
feat_importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

print(feat_importance_df)
# Plot top 15 features
plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=feat_importance_df.head(15), palette="viridis")
plt.title('Top 15 Feature Importances')
plt.tight_layout()
plt.show()


In [ ]:
# Applying Random Forest Classifier on thr selected Features only
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

X_selected = X_res[['V14', 'V12', 'V3', 'V4', 'V17', 'V2', 'V11', 'V10', 'V16']]

# Split the data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X_selected, y_res, test_size=0.2, random_state=42)

# Initialize and train Random Forest Classifier
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train, y_train)

# Predict on the test set
y_pred = rf_model.predict(X_test)

# Classification report
print("Random Forest Classification Report:")
print(classification_report(y_test, y_pred))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)

# Plot confusion matrix using seaborn heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=['Non-Fraud', 'Fraud'], yticklabels=['Non-Fraud', 'Fraud'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Random Forest Confusion Matrix')
plt.show()


In [ ]:
from sklearn.model_selection import cross_val_score

cv_scores = cross_val_score(rf_model, X_selected, y_res, cv=5, scoring='accuracy')
print("Cross-validated accuracy:", cv_scores.mean())


In [ ]:
# Applying Randomized search for HyperParameter Tuning (In this Case, Best parameters found:  {'max_depth': 17, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 170})
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint

param_dist = {
    'n_estimators': randint(100, 300),
    'max_depth': randint(5, 30),
    'min_samples_split': randint(2, 10),
    'min_samples_leaf': randint(1, 10)
}

random_search = RandomizedSearchCV(
    estimator=rf_model,
    param_distributions=param_dist,
    n_iter=30,  # 30 random combos only (instead of 324)
    cv=3,
    n_jobs=-1,
    verbose=2,
    random_state=42
)

random_search.fit(X_selected, y_res)

# Best parameters from RandomSearchCV
print("Best parameters found: ", random_search.best_params_)

# Best model from random search
best_rf_model = random_search.best_estimator_

# Evaluate the best model
y_pred_best = best_rf_model.predict(X_test)
print("Best Model Classification Report:")
print(classification_report(y_test, y_pred_best))

cm = confusion_matrix(y_test, y_pred_best)

# Plot confusion matrix using seaborn heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=['Non-Fraud', 'Fraud'], yticklabels=['Non-Fraud', 'Fraud'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Random Forest Confusion Matrix')
plt.show()

